In [ ]:

import os
import matplotlib.pyplot as plt
import gc
import torch
import sys
sys.path.insert(0, os.path.abspath(os.path.join("..")))
from pipeline.preprocessing import BackgroundEraser
import glob

from PIL import Image
from torchvision import transforms

import pandas as pd
import re as regex_module
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, fcluster

from ultralytics import YOLO

from pipeline.preprocessing import Repair, Normalizer, Augmentor
from pipeline.feature_learning.vae import VAE
from pipeline.feature_learning.vae_classifier import VAEClassifier
from pipeline.feature_learning.simclr import SimCLR
from pipeline.feature_learning.byol import BYOL
from pipeline.clustering import HierarchicalClustering
from pipeline.evaluation import EvaluationWithGroundTruth, EvaluationWithNoGroundTruth

# PIPELINE

## Configuration

Set `source_folder` and `run_id` here, then any section below can be run
on its own (fresh kernel included) as long as this cell has run first -
every step reads/writes under `pipeline_dir`, which is namespaced by
`run_id`, so nothing else in the notebook needs editing when you point at
new photos or start a new experiment run.

- **New photos added to `source_folder`?** Just re-run from the top with
  a new `run_id` - the old run's outputs stay untouched under their own
  `pipeline_dir`.
- **Want to jump straight to e.g. Clustering?** Run this cell, then the
  cells in that section alone - they all read their input from disk via
  the paths defined here, not from variables set by earlier sections.

In [2]:
# --- Edit these two for a new run ------------------------------------
source_folder = "/caa/Homes01/alaczko/Diploma/Diploma/data/raw"
run_id = "run_legion_bb"
# -----------------------------------------------------------------------

labels_path = "/caa/Homes01/alaczko/Diploma/Diploma/data/rocopot_results.csv"
test_image_path = "/caa/Homes01/alaczko/Diploma/Diploma/data/test3.png"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Every step's output lives under here, namespaced by run_id - switching
# run_id (or source_folder) is enough to start a fully separate run.
pipeline_dir = f"../data/pipeline/{run_id}"

erased_dir = f"{pipeline_dir}/PreprocessingStep/BackgroundEraser/"
normalized_dir = f"{pipeline_dir}/normalized" 
repaired_dir = f"{pipeline_dir}/repaired"
augmented_dir = f"{pipeline_dir}/augmented"

# Toggle: train VAE/VAEClassifier on the augmented set, or set this to
# repaired_dir to skip augmentation and train directly on the repaired images.
feature_learning_input_dir = augmented_dir

vae_checkpoint_path = f"{pipeline_dir}/vae.pt"
vae_features_csv = f"{pipeline_dir}/vae_features.csv"
simclr_checkpoint_path = f"{pipeline_dir}/simclr.pt"
simclr_features_csv = f"{pipeline_dir}/simclr_features.csv"
byol_checkpoint_path = f"{pipeline_dir}/byol.pt"
byol_features_csv = f"{pipeline_dir}/byol_features.csv"
dinov3_features_csv = f"{pipeline_dir}/dinov3_features.csv"
classifier_output_dir = f"{pipeline_dir}/classifier"

# Toggle: which feature set the Clustering section below clusters on.
features_csv_for_clustering = byol_features_csv
features_scaled_csv = f"{pipeline_dir}/features_scaled.csv"

roco_labels = pd.read_csv(labels_path)

In [3]:
roco_labels["TypeRev"].unique()

array(['pan', 'lid', 'pot', 'basin', 'jug', 'bottle', 'domestic amphora', 'jar', 'bowl', 'censer', 'dish-lid', 'base', 'knob', 'mortarium', 'dish', 'beaker', 'cencer', 'unguentarium', 'pitcher', nan, 'nd', 'handle', 'cooking stand', 'mortar', 'dolium', 'funnel', 'bedpan', 'basin '], dtype=object)

## Preprocessing

### Shard selection and  background cleaning

In [4]:
br = BackgroundEraser(
    input_path=source_folder,
    output_path=erased_dir,
    batch_size=128,
    num_workers=16,
    run_id=run_id,
)
#todo "detected 6171 shards, but erased len is 2800"
br.run()

Output path for step BackgroundEraser : ../data/pipeline/run_legion_bb/PreprocessingStep/BackgroundEraser
Detecting shards...


Detecting shards: 100%|██████████| 6273/6273 [02:44<00:00, 38.25image/s]


Detected 5621 shard(s), failed on 652
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/011.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/042.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/048.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/076.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/101.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/123.jpg
  - /caa/Homes01/alaczko/Diploma/Diploma/data/raw/00_Selection ML/1976_Schedivy Zivilstadt GB_Sektoren 1-2 bis 21-22/Sektor 1-2/152.jpg
  - /caa/H

Clearing images: 100%|██████████| 5621/5621 [10:05<00:00,  9.29it/s]


'../data/pipeline/run_legion_bb/PreprocessingStep/BackgroundEraser'

### Resize & Crop

In [5]:
crop_step = Normalizer(
    input_path=erased_dir,
    output_path=normalized_dir,
    background_color=(255, 255, 255),
    run_id=run_id,
)

crop_step.run()

Output path for step Normalizer : ../data/pipeline/run_legion_bb/normalized


Normalizing images:   4%|▍         | 233/5621 [00:12<04:37, 19.39image/s]


KeyboardInterrupt: 

### Enhancing Picture Quality

In [ ]:
repair_step = Repair(
    input_path=normalized_dir,
    output_path=repaired_dir,
    max_distance=10,
    output_thickness=2,
    run_id=run_id,
    min_component_size=6
)

repair_step.run()

### Data Augmentation

Optional: grows the (small) labelled dataset with randomly rotated/flipped/
jittered copies before training, to fight the overfitting seen earlier when
fine-tuning `VAEClassifier` end to end. Sits right after `Repair`, before
any training step.

In [ ]:
augment_step = Augmentor(
    input_path=repaired_dir,
    output_path=augmented_dir,
    num_augmentations=2,   # 2 extra randomly perturbed copies per image
    run_id=run_id,
)

augment_step.run()

Every Feature Learning cell below reads its input from
`feature_learning_input_dir` (set in Configuration) rather than a variable
from this section - it defaults to `augmented_dir`; set it to
`repaired_dir` there instead to skip augmentation.

## Feature Learning

### VAE

In [ ]:
vae = VAE(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)  # k=128 latent dimension
vae.to(device)

results = vae.run('train_model', epochs=500, lr=1e-5, batch_size=128)

torch.save(vae.state_dict(), vae_checkpoint_path)

In [ ]:
vae = VAE(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)
vae.to(device)

vae.load_state_dict(torch.load(vae_checkpoint_path, map_location=device))

results = vae.extract_features(batch_size=128)

In [ ]:
results

In [ ]:
results.to_csv(vae_features_csv, index=False)

#### Testing Feature Learning

In [ ]:
vae = VAE(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)
vae.to(device)

vae.load_state_dict(torch.load(vae_checkpoint_path, map_location=device))

#making sure the formula is correct
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

original_img = Image.open(test_image_path)
input_tensor = transform(original_img).unsqueeze(0).to(device) #adding batch dim

with torch.no_grad():
    reconstructed, mu, sigma = vae(input_tensor)

orig_np = input_tensor.squeeze().cpu().numpy()
recon_np = reconstructed.squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig_np, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(recon_np, cmap='gray', vmin=0, vmax=1)
axes[1].set_title("Reconstructed")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### SimCLR (contrastive feature learning, alternative to the VAE)

Trains the *same* `Encoder` architecture as the VAE (same `f`/`n`/`s`/`k`),
but with a contrastive objective instead of reconstruction: the encoder
learns to recognize that two differently-augmented views of the same
image belong together, and differ from every other image in the batch.
No labels needed.

Why try this alongside the VAE: since both use the identical backbone,
any difference in downstream clustering/classification accuracy between
the two isolates the *training objective* as the cause, not network
capacity - directly testing the hypothesis from the earlier discussion
that the VAE's reconstruction loss (not a lack of data) is what's
limiting its features' usefulness for classification.

In [ ]:
simclr = SimCLR(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)  # same architecture as the VAE
simclr.to(device)

simclr_results = simclr.run(
    "train_model",
    epochs=100,
    lr=1e-4,
    batch_size=128,
    temperature=0.5,
)

torch.save(simclr.state_dict(), simclr_checkpoint_path)

In [ ]:
simclr = SimCLR(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)
simclr.to(device)

simclr.load_state_dict(torch.load(simclr_checkpoint_path, map_location=device))

simclr_results = simclr.extract_features(batch_size=128)
simclr_results

In [ ]:
simclr_results.to_csv(simclr_features_csv, index=False)

To cluster on SimCLR features instead of the VAE's, set
`features_csv_for_clustering = simclr_features_csv` in Configuration
instead of `vae_features_csv`. Like DINOv3, SimCLR has no `sigma` (its
encoder is used deterministically here, no sampling) - so skip the
`df['sigma'] = ...` parsing line and the mu+sigma concatenation in the
scaling cell, and just standardize `df['features']` directly.

### BYOL (self-supervised, no negative pairs - alternative to SimCLR)

Like SimCLR, trains the same `SimCLREncoder` architecture to relate two
augmented views of the same image - but BYOL never contrasts against
*other* images at all, so there's no negative-pair pool and no
dependence on batch size for the loss to make sense (a real limitation
SimCLR has at this dataset's scale). Instead, BYOL keeps two copies of
the network: an "online" one trained normally, and a "target" one that's
only ever updated as an exponential moving average of the online
network's weights - the online network is trained to *predict* the
target network's view of the other augmented image.

Worth trying alongside SimCLR for the same reason SimCLR is worth trying
alongside the VAE: same backbone, different objective, so any difference
in downstream clustering quality isolates the objective as the cause.

In [ ]:
byol = BYOL(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)  # same encoder architecture as VAE/SimCLR
byol.to(device)

byol_results = byol.run(
    "train_model",
    epochs=100,
    lr=1e-4,
    batch_size=128,   # BYOL needs no negative-pair pool, so this can be smaller than SimCLR's
    early_stopping_patience=30
)

torch.save(byol.state_dict(), byol_checkpoint_path)

In [ ]:
byol = BYOL(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)
byol.to(device)

byol.load_state_dict(torch.load(byol_checkpoint_path, map_location=device))

byol_results = byol.extract_features(batch_size=128)
byol_results

In [ ]:
byol_results.to_csv(byol_features_csv, index=False)

To cluster on BYOL features instead, set
`features_csv_for_clustering = byol_features_csv` in Configuration. Like
SimCLR/DINOv3, BYOL has no `sigma` - skip the `df['sigma'] = ...` parsing
line and the mu+sigma concatenation in the scaling cell below, and just
standardize `df['features']` directly.

### DINOv3 (pretrained feature extractor, alternative to the VAE)

Instead of training an encoder from scratch, this uses Meta's DINOv3 - a
self-supervised Vision Transformer pretrained on ~1.7B natural images - as
a frozen feature extractor. No training happens here at all; every image
is just embedded once with the pretrained backbone.

Why try this: a VAE's encoder is only ever pushed to encode whatever makes
*pixel reconstruction* easy (see the earlier discussion on the encoder
being weak for classification) - DINO-style self-supervised training
optimizes for a much more discriminative objective, so its features tend
to transfer to downstream classification/clustering far better than plain
autoencoder features, even though DINOv3 was never shown an image of a
pottery sherd during its own training. It's a useful baseline for exactly
that reason: if DINOv3's *generic* features already separate your pottery
types better than the *purpose-built* VAE does, that's strong evidence the
VAE (or its BCE+KL training objective) - not the amount of data - is the
bottleneck.

**Before running this**: DINOv3 checkpoints are gated on Hugging Face.
Accept the license at the model page (e.g.
https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m) with the
account you'll authenticate with, then run `huggingface-cli login` once
(or set the `HF_TOKEN` environment variable) before executing the next
cell - otherwise the download will fail with a 403.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

# Swap for "facebook/dinov3-vits16-pretrain-lvd1689m" (smaller/faster) or
# "facebook/dinov3-vitl16-pretrain-lvd1689m" (larger/slower, likely better
# features) to compare backbone sizes.
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"

dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_NAME)
dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_NAME).to(device)
dinov3_model.eval()  # frozen - inference only, no training/fine-tuning here

In [ ]:
from pipeline.preprocessing.dataset import collect_image_paths


@torch.no_grad()
def extract_dinov3_features(image_paths, batch_size=32):
    """Embed each image with the frozen DINOv3 backbone's CLS token - the
    standard whole-image descriptor for ViT-style models, analogous to the
    VAE's `mu`. Returns a 'filename' + 'features' row per image, i.e. the
    same shape as `VAE.extract_features`'s output, so it can be pointed at
    directly from the Clustering section below by just swapping the CSV
    path - no other changes needed there.
    """
    filenames = []
    all_features = []

    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        # DINOv3 (like most ImageNet-pretrained ViTs) expects 3-channel
        # RGB input; convert("RGB") turns our grayscale line drawings into
        # a 3-channel image with all channels equal.
        images = [Image.open(p).convert("RGB") for p in batch_paths]

        inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
        outputs = dinov3_model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        filenames.extend(p.name for p in batch_paths)
        all_features.append(cls_embeddings.cpu())

    features = torch.cat(all_features, dim=0).numpy()
    return pd.DataFrame({"filename": filenames, "features": list(features)})


dinov3_image_paths = collect_image_paths(feature_learning_input_dir)
print(f"Found {len(dinov3_image_paths)} images for DINOv3 feature extraction")

dinov3_results = extract_dinov3_features(dinov3_image_paths, batch_size=128)
dinov3_results

In [ ]:
dinov3_results.to_csv(dinov3_features_csv, index=False)

To cluster on DINOv3 features instead of the VAE's, set
`features_csv_for_clustering = dinov3_features_csv` in Configuration
instead of `vae_features_csv`. One difference to account for: DINOv3 has
no `sigma` (it's deterministic, not a VAE) - so skip the
`df['sigma'] = ...` parsing line and the mu+sigma concatenation in the
scaling cell, and just standardize `df['features']` directly (single
`mu_matrix`-equivalent, no `hstack`).

### Classifier

In [ ]:
vae = VAE(input_path=feature_learning_input_dir, f=4, n=3, s=1, k=128, run_id=run_id)
vae.to(device)

vae.load_state_dict(torch.load(vae_checkpoint_path, map_location=device))

In [ ]:
classifier = VAEClassifier(
    input_path=feature_learning_input_dir,
    vae=vae,                         # a fentebb betanított VAE
    num_classes=28,
    labels_path=labels_path,
    freeze_encoder=True,             # először csak a classifier tanul
    output_path=classifier_output_dir,
    run_id=run_id,
).to(device)

history = classifier.run(
    epochs=200,
    lr=1e-4,
    batch_size=128,
    validation_size=0.2,
    num_workers=16,
)

history

### Clustering

In [ ]:
df = pd.read_csv(features_csv_for_clustering)

In [ ]:
import numpy as np
import ast

def parse_feature_string(s):
    if isinstance(s, str):
        cleaned = s.strip('[]')
        # tetszőleges mennyiségű whitespace -> egyetlen szóköz
        cleaned = regex_module.sub(r'\s+', ' ', cleaned).strip()
        return np.fromstring(cleaned, sep=' ')
    return np.asarray(s)

In [ ]:
df['features'] = df['features'].apply(parse_feature_string)

In [ ]:
X = np.vstack(df['features'].values)

# 2. inaktív / szinte nulla varianciájú dimenziók kiszűrése (opcionális, de ajánlott)
variances = X.var(axis=0)
active_dims = variances > 1e-3   # threshold-öt finomítsd a saját adatodra
X_active = X[:, active_dims]

# 3. standardizálás (dimenziónkénti z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_active)

df["features_non_scaled"] = df['features'] 

# 4. opcionálisan visszaírhatod a df-be, hogy egyben tudd kezelni
df['features'] = list(X_scaled)

df.to_csv(features_scaled_csv, index=False)

In [ ]:
df.head()

In [ ]:
clustering_step = HierarchicalClustering(
    input_path=features_scaled_csv,
    cluster_num=28,
    run_id=run_id,
)
# result= clustering_step.run()


#result = clustering_step.compare_all()

In [ ]:
result= clustering_step.run()

In [ ]:
import shutil
from pathlib import Path
clusters_output_dir = Path(pipeline_dir) / "clusters"
source_dir = Path(feature_learning_input_dir)
def resolve_image_path(filename):
    candidate = source_dir / filename
    if candidate.exists():
        return candidate
    # Robust to df["filename"] already having had its extension stripped
    # (see the base_id cell further down) - fall back to matching by stem.
    matches = list(source_dir.glob(f"{Path(filename).stem}.*"))
    return matches[0] if matches else None
saved, missing = 0, 0
for filename, cluster_id in zip(df["filename"], result):
    src_path = resolve_image_path(filename)
    if src_path is None:
        missing += 1
        continue
    cluster_dir = clusters_output_dir / f"cluster_{cluster_id}"
    cluster_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, cluster_dir / src_path.name)
    saved += 1
print(f"Saved {saved} images into {clusters_output_dir} ({len(set(result))} clusters), {missing} missing.")

In [ ]:
np.unique(result, return_counts=True)

#### Silhouette Score (no ground truth needed)

In [ ]:
# Unsupervised evaluation: uses only the feature vectors + predicted cluster
# labels, so it works even before/without matching against ground-truth types.
silhouette_step = EvaluationWithNoGroundTruth(features=clustering_step.vectors, y_pred=result)
silhouette_step.run()

In [ ]:
df["pred"] = list(result)

In [ ]:
df["filename"] = df["filename"].str.rsplit(".", n=1).str[0]

# Augmentor saves augmented copies as "<original>_aug0", "<original>_aug1",
# etc., which have no matching row in roco_labels (only the original ID
# does). base_id strips that suffix back off, so every augmented copy is
# looked up under its source image's label - correct, since augmentation
# doesn't change what type the sherd actually is.
df["base_id"] = df["filename"].str.replace(r"_aug\d+$", "", regex=True)

In [ ]:
new_df = pd.merge(
    df,
    roco_labels,
    left_on="base_id",
    right_on="ID",
    how="inner"
)

In [ ]:
new_df = new_df.dropna(subset=["pred", "TypeRev"]).reset_index(drop=True)

In [ ]:
new_df

In [ ]:
eval_step = EvaluationWithGroundTruth(y_true=new_df["TypeRev"].values, y_pred=new_df["pred"].values)

In [ ]:
eval_step.run()

In [ ]:
features_path = features_scaled_csv

cluster_numbers = range(3, 60)
metric_results = []

# A features.csv sorrendjében találhatók a klaszterezés bemenetei
features_df = pd.read_csv(features_path)

features_df["filename"] = (
    features_df["filename"]
    .astype(str)
    .str.strip()
    .str.rsplit(".", n=1)
    .str[0]
)

# See the base_id comment above: augmented copies ("<original>_aug0", ...)
# need this suffix stripped to find their source image's label.
features_df["base_id"] = features_df["filename"].str.replace(r"_aug\d+$", "", regex=True)

In [ ]:
for cluster_num in cluster_numbers:
    clustering_step = HierarchicalClustering(
        input_path=features_path,
        cluster_num=cluster_num,
        run_id=run_id,
    )

    predicted_labels = clustering_step.run()

    # Unsupervised metric: doesn't need the ground-truth join below, so it's
    # computed on every sample that went into clustering.
    silhouette_step = EvaluationWithNoGroundTruth(
        features=clustering_step.vectors, y_pred=predicted_labels
    )
    silhouette_step.run()

    predictions_df = features_df[["filename", "base_id"]].copy()
    predictions_df["pred"] = predicted_labels

    # Az új klasztercímkék hozzárendelése a meglévő dataframe-hez
    current_df = pd.merge(
        predictions_df,
        roco_labels,
        left_on="base_id",
        right_on="ID",
        how="inner"
    )

    current_df = current_df.dropna(subset=["pred", "TypeRev"]).reset_index(drop=True)

    y_true = current_df["TypeRev"].to_numpy()
    y_pred = current_df["pred"].to_numpy()


    eval_step = EvaluationWithGroundTruth(y_true=y_true, y_pred=y_pred)

    eval_step.run()

    metric_results.append({
        "cluster_num": cluster_num,
        "ACC": eval_step.acc,
        "ARI": eval_step.ari,
        "NMI": eval_step.nmi,
        "Silhouette": silhouette_step.silhouette,
        "sample_count": len(current_df),
        "predicted_cluster_count": current_df["pred"].nunique(),
    })

metrics_df = pd.DataFrame(metric_results)
metrics_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.plot(
    metrics_df["cluster_num"],
    metrics_df["ACC"],
    marker="o",
    label="ACC"
)

ax1.plot(
    metrics_df["cluster_num"],
    metrics_df["ARI"],
    marker="o",
    label="ARI"
)

ax1.plot(
    metrics_df["cluster_num"],
    metrics_df["NMI"],
    marker="o",
    label="NMI"
)

ax1.axvline(
    28,
    color="red",
    linestyle="--",
    label="28 archaeological labels"
)

ax1.set_xlabel("Number of predicted clusters")
ax1.set_ylabel("Metric value (ACC / ARI / NMI)")
ax1.set_xticks(range(10, 61, 5))
ax1.grid(alpha=0.3)

# Silhouette score has a different range ([-1, 1], unsupervised - no ground
# truth involved) than ACC/ARI/NMI, so it gets its own y-axis rather than
# being squashed onto the same 0-1 scale.
ax2 = ax1.twinx()
ax2.plot(
    metrics_df["cluster_num"],
    metrics_df["Silhouette"],
    marker="s",
    color="black",
    linestyle="--",
    label="Silhouette",
)
ax2.set_ylabel("Silhouette score")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

plt.title("Clustering metrics by number of clusters")
plt.tight_layout()
plt.show()

In [ ]:
new_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# df-ben legyen egy 'pred' (predikált cluster) és egy 'true_label' (ground truth) oszlop
contingency = pd.crosstab(new_df['TypeRev'], new_df['pred'])

plt.figure(figsize=(14, 10))
sns.heatmap(contingency, annot=False, cmap='viridis', cbar_kws={'label': 'count'})
plt.xlabel('Predicted cluster')
plt.ylabel('Ground truth type')
plt.title('Cluster-Ground Truth contingency matrix')
plt.tight_layout()
plt.show()